In [ ]:
from Models.BE_model import BoundaryEstimationModel
from Models.SC_model import StimulusCategoryModel

import numpy as np
from scipy import stats

class ModelComparison:
    """Utilities for comparing BE and SC models."""
    
    @staticmethod
    def vuong_test(ll_model1, ll_model2):
        """
        Vuong's test for comparing non-nested models.
        
        Args:
            ll_model1: array of per-trial log-likelihoods for model 1
            ll_model2: array of per-trial log-likelihoods for model 2
        
        Returns:
            vuong_stat: positive favours model 1, negative favours model 2
            p_value: two-tailed p-value
        """        
        n = len(ll_model1)
        lr = ll_model1 - ll_model2
        
        m = np.mean(lr)
        s = np.std(lr, ddof=1)
        
        if s < 1e-10:
            return 0.0, 1.0
        
        vuong_stat = np.sqrt(n) * m / s
        p_value = 2 * (1 - stats.norm.cdf(np.abs(vuong_stat)))
        
        return vuong_stat, p_value
    
    @staticmethod
    def compare(stimuli, categories, observed_choices, no_response=None,
                BE_fixed_params=None, SC_fixed_params=None):
        """
        Compare BE and SC models on same data.
        
        Returns:
            dict with fitted models, parameters, likelihoods, AIC, Vuong test
        """
        # Fit both models
        BE_model, BE_results = BoundaryEstimationModel.fit(
            stimuli, categories, observed_choices, no_response,
            fixed_params=BE_fixed_params
        )
        
        SC_model, SC_results = StimulusCategoryModel.fit(
            stimuli, categories, observed_choices, no_response,
            fixed_params=SC_fixed_params
        )
        
        # Get per-trial log-likelihoods
        BE_model_fresh = BoundaryEstimationModel(**BE_results['params'])
        SC_model_fresh = StimulusCategoryModel(**SC_results['params'])
        
        ll_BE = BE_model_fresh.compute_per_trial_ll(
            stimuli, categories, observed_choices, no_response
        )
        ll_SC = SC_model_fresh.compute_per_trial_ll(
            stimuli, categories, observed_choices, no_response
        )
        
        # Vuong's test
        vuong_stat, vuong_p = ModelComparison.vuong_test(ll_BE, ll_SC)
        
        # Determine winner
        if vuong_p < 0.05:
            winner = 'BE' if vuong_stat > 0 else 'SC'
        else:
            winner = 'tie'
        
        return {
            'BE_model': BE_model,
            'SC_model': SC_model,
            'BE_results': BE_results,
            'SC_results': SC_results,
            'll_BE': ll_BE,
            'll_SC': ll_SC,
            'nll_BE': BE_results['nll'],
            'nll_SC': SC_results['nll'],
            'aic_BE': BE_results['aic'],
            'aic_SC': SC_results['aic'],
            'delta_aic': SC_results['aic'] - BE_results['aic'],
            'vuong_stat': vuong_stat,
            'vuong_p': vuong_p,
            'winner': winner
        }
    
    @staticmethod
    def compare_sessions(sessions, BE_fixed_params=None, SC_fixed_params=None,
                         verbose=True):
        """Compare models across multiple sessions."""
        results = []
        
        for i, session in enumerate(sessions):
            if verbose:
                print(f"Comparing session {i+1}/{len(sessions)}...", end=' ')
            
            try:
                comp = ModelComparison.compare(
                    session['stimuli'],
                    session['categories'],
                    session['choices'],
                    session.get('no_response', None),
                    BE_fixed_params=BE_fixed_params,
                    SC_fixed_params=SC_fixed_params
                )
                comp['session_idx'] = i
                comp['success'] = True
                results.append(comp)
                
                if verbose:
                    print(f"Winner: {comp['winner']} (ΔAIC={comp['delta_aic']:.1f})")
            
            except Exception as e:
                if verbose:
                    print(f"FAILED: {e}")
                results.append({'session_idx': i, 'success': False})
        
        return results
    
    @staticmethod
    def summarise_comparison(results):
        """Summarise model comparison across sessions."""
        successful = [r for r in results if r.get('success', False)]
        
        n_BE_wins = sum(1 for r in successful if r['winner'] == 'BE')
        n_SC_wins = sum(1 for r in successful if r['winner'] == 'SC')
        n_ties = sum(1 for r in successful if r['winner'] == 'tie')
        
        delta_aics = [r['delta_aic'] for r in successful]
        
        return {
            'n_sessions': len(successful),
            'n_BE_wins': n_BE_wins,
            'n_SC_wins': n_SC_wins,
            'n_ties': n_ties,
            'mean_delta_aic': np.mean(delta_aics),
            'std_delta_aic': np.std(delta_aics),
            'fraction_BE': n_BE_wins / len(successful) if successful else 0
        }

In [ ]:
# load data

In [ ]:
# Load your data (you'll need to adapt this to your data format)
stimuli = session_data['stimuli']           # array of stimulus values
categories = session_data['categories']      # array of true categories (0/1)
observed_choices = session_data['choices']   # array of animal choices (0/1)
no_response = session_data['no_response']    # boolean array

In [ ]:
# 1. Compare models (single session)
comparison = ModelComparison.compare(stimuli, categories, observed_choices)
print(f"Winner: {comparison['winner']}, ΔAIC: {comparison['delta_aic']:.2f}")

# 2. Compare models (all sessions)
comparisons = ModelComparison.compare_sessions(sessions)
summary = ModelComparison.summarise_comparison(comparisons)
print(f"BE wins: {summary['n_BE_wins']}/{summary['n_sessions']}")